# Luxury Fragrance Dataset — Data Cleaning

This notebook transforms the raw fragrance marketplace dataset into a cleaner and more consistent analytical dataset.

The cleaning decisions are based on observations identified during the exploratory analysis.

Objectives:

- Standardize categorical values
- Normalize brand names and identify non-brand values
- Separate obvious product information from contaminated brand fields
- Standardize fragrance type categories
- Preserve the original marketplace information where possible
- Identify and flag anomalous prices rather than automatically removing them
- Prepare a reliable dataset for enrichment and further analysis

In [0]:
# Load the project-owned marketplace table.
# This table contains the permanent technical ID created in the EDA/base-data notebook.

from pyspark.sql import functions as F

df = spark.table("main.fragrance_analytics.marketplace_base")

print(f"Rows: {df.count():,}")
print(f"Columns: {len(df.columns)}")

df.printSchema()

display(df.limit(20))

In [0]:
# Preserve the original marketplace brand values before applying any cleaning.
# This allows us to compare the raw seller-entered values with the cleaned version.
df = df.withColumn("brand_raw", F.col("brand"))

# Standardize basic formatting and special characters:
# - remove leading and trailing whitespace
# - collapse repeated internal whitespace
# - normalize accented Latin characters to their base letters
# - normalize curly apostrophes to the standard apostrophe
# - convert everything to lowercase
df = df.withColumn(
    "brand_normalized",
    F.lower(
        F.regexp_replace(
            F.regexp_replace(
                F.regexp_replace(
                    F.regexp_replace(
                        F.regexp_replace(
                            F.regexp_replace(
                                F.regexp_replace(
                                    F.regexp_replace(
                                        F.trim(F.col("brand")),
                                        r"[ÉÈÊËéèêë]",
                                        "e"
                                    ),
                                    r"[ÁÀÂÄÃÅáàâäãå]",
                                    "a"
                                ),
                                r"[ÓÒÔÖÕóòôöõ]",
                                "o"
                            ),
                            r"[ÍÌÎÏíìîï]",
                            "i"
                        ),
                        r"[ÚÙÛÜúùûü]",
                        "u"
                    ),
                    r"[Ññ]",
                    "n"
                ),
                r"[Çç]",
                "c"
            ),
            "’",
            "'"
        )
    )
)

# Collapse repeated internal whitespace after character normalization.
df = df.withColumn(
    "brand_normalized",
    F.regexp_replace(
        F.trim(F.col("brand_normalized")),
        r"\s+",
        " "
    )
)

# Inspect the raw and normalized values side by side.
display(
    df.select(
        "brand_raw",
        "brand_normalized"
    )
    .distinct()
    .orderBy("brand_raw")
)

In [0]:
# Remove listings where the brand field is a generic descriptor rather than an identifiable fragrance brand.
df = df.filter(
    ~F.col("brand_normalized").isin(
        "assorted",
        "eau de parfum"
    )
)

In [0]:
# Remove listings where the brand cannot be reliably identified.

df = df.filter(
    ~F.col("id").isin(
        1001,
        1026,
        1027,
        1071,
        1075,
        1111,
        1143,
        1225,
        1397,
        1442,
        1588,
        1619,
        1667,
        2012,
        2017,
        2591,
        2869
    )
)

In [0]:
# Create the canonical analytical brand field.
# Brand names are kept in lowercase for consistent matching and analysis.
# The original marketplace value remains available in brand_raw.

df = df.withColumn(
    "brand_clean",

    # Alain Aregon
    # "1 Billion" is a fragrance name, not the brand.
    F.when(
        F.col("brand_normalized").contains("1 billion"),
        "alain aregon"
    )

    # Amouage
    .when(
        F.col("brand_normalized").contains("amouage"),
        "amouage"
    )

    # Antonio Puig
    .when(
        F.col("brand_normalized").contains("antonio puig"),
        "antonio puig"
    )

    # Aramis
    .when(
        F.col("brand_normalized").contains("aramis"),
        "aramis"
    )

    # Atelier Cologne
    .when(
        F.col("brand_normalized") == "atelier",
        "atelier cologne"
    )

    # Atelier Rebul
    .when(
        F.col("brand_normalized").rlike(
            r"atelier\s+(bosphorus\s+istanbul|istanbul|pera|rebul\s+1895)"
        ),
        "atelier rebul"
    )

    # Axis
    .when(
        F.col("brand_normalized").contains("axis"),
        "axis"
    )

    # Balmain
    .when(
        F.col("brand_normalized").contains("balmain"),
        "balmain"
    )

    # Bond No. 9
    .when(
        F.col("brand_normalized").rlike(r"bond\s*(9|n9|no\.?\s*9)"),
        "bond no. 9"
    )

    # Borsalino
    .when(
        F.col("brand_normalized").contains("borsalino"),
        "borsalino"
    )

    # Borsari
    .when(
        F.col("brand_normalized").contains("borsari"),
        "borsari"
    )

    # Bottega Veneta
    .when(
        F.col("brand_normalized").contains("bottega veneta"),
        "bottega veneta"
    )

    # Bourjois
    .when(
        F.col("brand_normalized").contains("bourjoi") |
        F.col("brand_normalized").contains("bourjois"),
        "bourjois"
    )

    # Burberry
    .when(
        F.col("brand_normalized").contains("burberry"),
        "burberry"
    )

    # Bvlgari
    .when(
        F.col("brand_normalized").contains("bvlgari"),
        "bvlgari"
    )

    # Caron
    .when(
        F.col("brand_normalized").contains("caron"),
        "caron"
    )

    # Cartier
    .when(
        F.col("brand_normalized").contains("cartier"),
        "cartier"
    )

    # Calvin Klein
    .when(
        F.col("brand_normalized").contains("calvin kleinn"),
        "calvin klein"
    )

    # Carolina Herrera
    # "212 Perfume" is a fragrance name associated with Carolina Herrera.
    .when(
        F.col("brand_normalized").contains("carolina herrera") |
        F.col("brand_normalized").contains("212 perfume"),
        "carolina herrera"
    )

    # Chanel
    .when(
        F.col("brand_normalized").contains("les exclusifs de chanel"),
        "chanel"
    )

    # Charles Revson
    .when(
        F.col("brand_normalized").contains("ultima ii"),
        "charles revson"
    )

    # Chloé
    .when(
        F.col("brand_normalized").contains("chloe") |
        F.col("brand_normalized").contains("chloé"),
        "chloe"
    )

    # Clive Christian
    .when(
        F.col("brand_normalized").rlike(r"clive[\s-]+christian"),
        "clive christian"
    )

    # Crabtree & Evelyn
    .when(
        F.col("brand_normalized").contains("crabtree evelyn"),
        "crabtree & evelyn"
    )

    # Creed
    .when(
        F.col("brand_normalized").contains("creed"),
        "creed"
    )

    # Davidoff
    .when(
        F.col("brand_normalized").contains("davidoff"),
        "davidoff"
    )

    # Dior
    .when(
        F.col("brand_normalized").contains("dior"),
        "dior"
    )

    # Dolce & Gabbana
    # Normalize spacing and ampersand variations.
    .when(
        F.col("brand_normalized").rlike(r"dolce\s*(?:&\s*)?gabbana"),
        "dolce&gabbana"
    )

    # Donna Karan
    .when(
        F.col("brand_normalized").contains("donna karan"),
        "donna karan"
    )

    # Dsquared2
    .when(
        F.col("brand_normalized").contains("dsquared2"),
        "dsquared2"
    )

    # Electimuss
    .when(
        F.col("brand_normalized").contains("electimuss"),
        "electimuss"
    )

    # Ermenegildo Zegna
    .when(
        F.col("brand_normalized").contains("ermenegildo zegina") |
        F.col("brand_normalized").contains("ermenegildo zegna"),
        "ermenegildo zegna"
    )

    # Escada
    .when(
        F.col("brand_normalized").contains("escada"),
        "escada"
    )

    # Estée Lauder
    .when(
        F.col("brand_normalized").contains("estee lauder") |
        F.col("brand_normalized").contains("estée lauder"),
        "estee lauder"
    )

    # Fabergé
    .when(
        F.col("brand_normalized").contains("fabergé") |
        F.col("brand_normalized").contains("faberge"),
        "faberge"
    )

    # Filthy Rich
    .when(
        F.col("brand_normalized").contains("filthy rich"),
        "filthy rich dubai"
    )

    # Frederic Malle Editions de Parfums
    .when(
        F.col("brand_normalized").contains("fredric malle") |
        F.col("brand_normalized").contains("frederic malle"),
        "frederic malle editions de parfums"
    )

    # Floris
    .when(
        F.col("brand_normalized").contains("floris"),
        "floris"
    )

    # Fragrance One
    # Marketplace value contains the literal "+": "fragrance +one".
    .when(
        F.col("brand_normalized").contains("fragrance +one"),
        "fragrance one"
    )

    # Giorgio Armani
    .when(
        F.col("brand_normalized").contains("armani"),
        "giorgio armani"
    )

    # Givenchy
    .when(
        F.col("brand_normalized").contains("givenchy"),
        "givenchy"
    )

    # Gres
    .when(
        F.col("brand_normalized").contains("grès") |
        F.col("brand_normalized").contains("gres"),
        "gres"
    )

    # Gucci
    .when(
        F.col("brand_normalized").contains("guccii") |
        F.col("brand_normalized").contains("gucci"),
        "gucci"
    )

    # Guerlain
    .when(
        F.col("brand_normalized").contains("guerlain"),
        "guerlain"
    )

    # Hermès
    .when(
        F.col("brand_normalized").contains("terre d'hermes"),
        "hermes"
    )

    # Hugo Boss
    .when(
        F.col("brand_normalized").contains("boss"),
        "hugo boss"
    )

    # Issey Miyake
    .when(
        F.col("brand_normalized").contains("le feu d'issey") |
        F.col("brand_normalized").contains("issey miyake") |
        F.col("brand_normalized").contains("issey miyakee"),
        "issey miyake"
    )

    # Initio Parfums Prives
    .when(
        F.col("brand_normalized").contains("initio"),
        "initio parfums prives"
    )

    # Jean Paul Gaultier
    .when(
        F.col("brand_normalized").contains("jean paul") |
        F.col("brand_normalized").contains("jean paul gaultier"),
        "jean paul gaultier"
    )

    # Joop!
    .when(
        F.col("brand_normalized").contains("joop!"),
        "joop!"
    )

    # Kiton
    .when(
        F.col("brand_normalized").contains("kiton"),
        "kiton"
    )

    # Lacoste
    .when(
        F.col("brand_normalized").contains("lacoste"),
        "lacoste fragrances"
    )

    # Lancôme
    .when(
        F.col("brand_normalized").contains("magie noire"),
        "lancome"
    )

    # MAC
    .when(
        F.col("brand_normalized").contains("m·a·c") |
        F.col("brand_normalized").contains("m.a.c"),
        "mac"
    )

    # Marc-Antoine Barrois
    .when(
        F.col("brand_normalized").contains("marc antoine"),
        "marc-antoine barrois"
    )

    # Mary Kay
    .when(
        F.col("brand_normalized").contains("mary kay"),
        "mary kay"
    )

    # Maison Francis Kurkdjian
    .when(
        F.col("brand_normalized").contains("masion francis kurkdjian paris") |
        F.col("brand_normalized").contains("maison francis kurkdjian"),
        "maison francis kurkdjian"
    )

    # Michael Kors
    .when(
        F.col("brand_normalized").contains("michael korrs") |
        F.col("brand_normalized").contains("michael korss"),
        "michael kors"
    )

    # Molinard
    .when(
        F.col("brand_normalized").contains("molinard"),
        "molinard"
    )

    # Mugler
    .when(
        F.col("brand_normalized").contains("thierry mugler"),
        "mugler"
    )

    # Olivina Napa Valley
    .when(
        F.col("brand_normalized").contains("cielo napa valley"),
        "olivina napa valley"
    )

    # Paloma Picasso
    .when(
        F.col("brand_normalized").contains("paloma piccaso") |
        F.col("brand_normalized").contains("paloma picasso"),
        "paloma picasso"
    )

    # Penhaligon's
    .when(
        F.col("brand_normalized").contains("penhaligons"),
        "penhaligon's"
    )

    # Prada
    .when(
        F.col("brand_normalized").contains("prada"),
        "prada"
    )


    # Rabanne
    .when(
        F.col("brand_normalized").contains("paco rabanne"),
        "rabanne"
    )

    # Ralph Lauren
    .when(
        F.col("brand_normalized").contains("ralph lauren"),
        "ralph lauren"
    )

    # Roberto Cavalli
    .when(
        F.col("brand_normalized").contains("roberto cavalli"),
        "roberto cavalli"
    )

    # Roja Dove
    .when(
        F.col("brand_normalized").contains("roja"),
        "roja dove"
    )

    # Ruehl No.925
    .when(
        F.col("brand_normalized").contains("ruehl"),
        "ruehl no.925"
    )

    # Salvador Dali
    .when(
        F.col("brand_normalized").contains("dali haute parfumerie"),
        "salvador dali"
    )

    # Stella McCartney
    .when(
        F.col("brand_normalized").contains("stella mccartny"),
        "stella mccartney"
    )

   # The Gate Fragrance(s) → The Gate Fragrances Paris
    .when(
        F.col("brand_normalized").isin(
            "the gate fragrance",
            "the gate fragrances"
        ),
        "the gate fragrances paris"
    )

    # Tiffany
    .when(
        F.col("brand_normalized").contains("tiffany"),
        "tiffany"
    )

    # Tiziana Terenzi
    .when(
        F.col("brand_normalized").contains("tiziana-terenzi"),
        "tiziana terenzi"
    )

    # Tom Ford
    .when(
        F.col("brand_normalized").contains("tf"),
        "tom ford"
    )

    # Trump
    .when(
        F.col("brand_normalized").contains("donald trump"),
        "trump"
    )

    # Trussardi
    .when(
        F.col("brand_normalized").contains("trussardi"),
        "trussardi"
    )

    # Valentino
    .when(
        F.col("brand_normalized").contains("uomo noir absolu"),
        "valentino"
    )

    # Van Cleef & Arpels
    .when(
        F.col("brand_normalized").contains("vancleef & arpels") |
        F.col("brand_normalized").contains("van cleef"),
        "van cleef & arpels"
    )

    # Versace
    .when(
        F.col("brand_normalized").contains("versace"),
        "versace"
    )

    # Victoria's Secret
    .when(
        F.col("brand_normalized").contains("victoria's secret"),
        "victoria's secret"
    )

    # Viktor&Rolf
    .when(
        F.col("brand_normalized").contains("victor & rolf") |
        F.col("brand_normalized").contains("viktor & rolf"),
        "viktor&rolf"
    )

    # Western Valley Avenue London
    .when(
        F.col("brand_normalized").contains("western valley"),
        "western valley avenue london"
    )

    # Yves Saint Laurent
    .when(
        F.col("brand_normalized").contains("yves saint laurent") |
        F.col("brand_normalized").contains("yves saint laurente") |
        F.col("brand_normalized").contains("yves st laurent") |
        F.col("brand_normalized").contains("ysl"),
        "yves saint laurent"
    )

    # YZY
    .when(
        F.col("brand_normalized").contains("yzy perfumes"),
        "yzy"
    )

    # Keep the normalized marketplace value until a specific
    # business-rule mapping has been defined.
    .otherwise(F.col("brand_normalized"))
)


# Inspect raw, normalized, and canonical brand values side by side.
display(
    df.select(
        "brand_raw",
        "brand_normalized",
        "brand_clean"
    )
    .distinct()
    .orderBy("brand_clean", "brand_raw")
)

In [0]:
# Apply brand corrections for specific marketplace listings identified by permanent ID.

df = df.withColumn(
    "brand_clean",
    F.when(F.col("id") == 53, "melanie martinez")
    .when(F.col("id") == 64, "melanie martinez")
    .when(F.col("id") == 161, "guerlain")
    .when(F.col("id") == 190, "melanie martinez")
    .when(F.col("id") == 206, "melanie martinez")
    .when(F.col("id") == 295, "melanie martinez")
    .when(F.col("id") == 384, "jean d'albret")
    .when(F.col("id") == 386, "melanie martinez")
    .when(F.col("id") == 396, "melanie martinez")
    .when(F.col("id") == 946, "mugler")
    .when(F.col("id") == 1019, "the boring company")
    .when(F.col("id") == 1058, "sapil")
    .when(F.col("id") == 1059, "the boring company")
    .when(F.col("id") == 1060, "sapil")
    .when(F.col("id") == 1065, "sapil")
    .when(F.col("id") == 1208, "creed")
    .when(F.col("id") == 1227, "armaf")
    .when(F.col("id") == 1265, "hermes")
    .when(F.col("id") == 1299, "armaf")
    .when(F.col("id") == 1300, "tiffany")
    .when(F.col("id") == 1388, "the boring company")
    .when(F.col("id") == 1542, "creed")
    .when(F.col("id") == 1609, "the boring company")
    .when(F.col("id") == 1625, "junaid jamshed")
    .when(F.col("id") == 1637, "junaid jamshed")
    .when(F.col("id") == 1746, "tom ford")
    .when(F.col("id") == 2250, "parfums de marly")
    .when(F.col("id") == 2326, "the boring company")
    .when(F.col("id") == 2338, "neesh perfumes")
    .when(F.col("id") == 2421, "the boring company")
    .when(F.col("id") == 2565, "giorgio armani")
    .when(F.col("id") == 2576, "the boring company")
    .when(F.col("id") == 2763, "bond no. 9")
    .when(F.col("id") == 2862, "particle")
    .when(F.col("id") == 2875, "jean paul gaultier")
    .when(F.col("id") == 2920, "bond no. 9")
    .when(F.col("id") == 2932, "bond no. 9")
    .when(F.col("id") == 2956, "the boring company")
    .when(F.col("id") == 2039, "neesh perfumes")
    .otherwise(F.col("brand_clean"))
)

# Verify the ID-based corrections.
display(
    df.filter(
        F.col("id").isin(
            53,
            64,
            161,
            190,
            206,
            295,
            384,
            386,
            396,
            946,
            1019,
            1058,
            1059,
            1060,
            1065,
            1208,
            1227,
            1265,
            1299,
            1300,
            1388,
            1542,
            1609,
            1625,
            1637,
            1746,
            2039,
            2250,
            2326,
            2338,
            2421,
            2565,
            2576,
            2763,
            2862,
            2875,
            2920,
            2932,
            2956
        )
    )
    .select(
        "id",
        "brand_raw",
        "brand_normalized",
        "brand_clean",
        "title"
    )
    .orderBy("id")
)

In [0]:
display(
    df.select("brand_clean")
      .distinct()
      .orderBy("brand_clean")
)

In [0]:
# Investigate suspicious brand values before deciding whether to normalize or remove them.
display(
    df.filter(
        F.col("brand_normalized").isin(
            "various - see photos",
            "leggi tutto f-collections",
            "grasse concentration fragrance gfc",
            "f collections",
            "does not apply",
            "diamond collection",
            "dearmusk.com",
            "cologne",
            "no brand",
            "unbranded",
            "undisclosed"
        )
    )
    .select(
        "id",
        "brand_raw",
        "brand_normalized",
        "brand_clean",
        "title",
        "type",
        "condition",
        "price",
        "priceWithCurrency",
        "seller",
        "itemLocation",
        "shipsTo",
        "lastUpdated",
        "perfumeType"
    )
    .orderBy("brand_normalized", "title")
)

In [0]:
# Final validation checks
# Check for missing marketplace brand values and measure the reduction
# in distinct brand values after cleaning.

# Check whether the original marketplace brand field contains null values.
null_brand_rows = (
    df.filter(F.col("brand_raw").isNull())
      .select(
          "id",
          "brand_raw",
          "brand_normalized",
          "brand_clean",
          "title"
      )
      .orderBy("id")
)

display(null_brand_rows)

null_brand_count = null_brand_rows.count()

print(f"Null brand_raw values: {null_brand_count:,}")


# Compare the number of distinct brand values before and after cleaning.
# This shows whether the normalization and canonicalization rules
# reduce fragmentation in the marketplace brand field.

raw_brand_count = (
    df.filter(F.col("brand_raw").isNotNull())
      .select("brand_raw")
      .distinct()
      .count()
)

clean_brand_count = (
    df.filter(F.col("brand_clean").isNotNull())
      .select("brand_clean")
      .distinct()
      .count()
)

print(f"Unique brand values before cleaning: {raw_brand_count:,}")
print(f"Unique brand values after cleaning: {clean_brand_count:,}")
print(f"Reduction in unique brand values: {raw_brand_count - clean_brand_count:,}")